# 🇵🇰 Urdu LLM — Fine-tuning Llama 3 on Pakistani Corpus

**Fine-tunes Llama 3.2-1B on a Pakistani Urdu corpus using LoRA (QLoRA)**

Dataset covers:
- 📰 Urdu news articles
- 📚 Urdu literature & poetry
- 🕌 Islamic/religious text
- 🏛️ Pakistani history & culture
- 💬 Code-switching (Urdu+English)

**Runtime: GPU (T4) | Time: ~30-45 mins**

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q sentencepiece protobuf huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# ── Step 2: Login to HuggingFace (to push model) ──────────────────────────────
from huggingface_hub import notebook_login
notebook_login()
# Enter your HuggingFace token when prompted
# Get free token at: https://huggingface.co/settings/tokens

In [ ]:
# ── Step 3: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── Step 4: Build Pakistani Urdu Dataset ──────────────────────────────────────
# We combine multiple sources into a single instruction-following dataset

URDU_DATA = [
    # ── Urdu Greetings & Culture ──────────────────────────────────────────────
    {"instruction": "آداب کا صحیح جواب کیا ہے؟", "output": "آداب کا جواب 'آداب عرض' یا 'خوش آمدید' سے دیا جاتا ہے۔ یہ اردو میں احترام کا اظہار ہے۔"},
    {"instruction": "پاکستان کا قومی ترانہ کب لکھا گیا؟", "output": "پاکستان کا قومی ترانہ حفیظ جالندھری نے لکھا اور اسے 1954 میں سرکاری طور پر منظور کیا گیا۔"},
    {"instruction": "اردو زبان کی تاریخ بتائیں", "output": "اردو زبان ہندوستانی برصغیر میں پیدا ہوئی۔ یہ فارسی، عربی، ترکی اور مقامی زبانوں کا حسین امتزاج ہے۔ مغل دور میں یہ دربار کی زبان بنی اور آج پاکستان کی قومی زبان ہے۔"},
    {"instruction": "پاکستان میں عید کیسے منائی جاتی ہے؟", "output": "پاکستان میں عید الفطر اور عید الاضحی بڑے جوش و خروش سے منائی جاتی ہے۔ لوگ نئے کپڑے پہنتے ہیں، عید کی نماز پڑھتے ہیں، گلے ملتے ہیں اور مٹھائیاں تقسیم کرتے ہیں۔"},
    {"instruction": "لاہور کو کیوں دل پاکستان کہتے ہیں؟", "output": "لاہور کو دل پاکستان اس لیے کہا جاتا ہے کیونکہ یہ پاکستان کا ثقافتی، تاریخی اور تعلیمی مرکز ہے۔ یہاں قلعہ لاہور، بادشاہی مسجد اور متعدد تاریخی مقامات ہیں۔"},

    # ── Islamic Knowledge ────────────────────────────────────────────────────
    {"instruction": "نماز کے اوقات کتنے ہیں؟", "output": "نماز کے پانچ اوقات ہیں: فجر، ظہر، عصر، مغرب اور عشاء۔ یہ پانچ نمازیں ہر مسلمان پر فرض ہیں۔"},
    {"instruction": "رمضان المبارک کی فضیلت بیان کریں", "output": "رمضان المبارک اسلام کا مقدس مہینہ ہے جس میں قرآن کریم نازل ہوا۔ اس مہینے میں روزہ رکھنا فرض ہے، تراویح پڑھی جاتی ہے اور لیلۃ القدر کی تلاش کی جاتی ہے جو ہزار مہینوں سے بہتر ہے۔"},
    {"instruction": "حج کیا ہے اور کس پر فرض ہے؟", "output": "حج اسلام کا پانچواں رکن ہے۔ ہر وہ مسلمان جو صاحب استطاعت ہو، اس پر زندگی میں ایک بار حج ادا کرنا فرض ہے۔ حج ذوالحجہ کے مہینے میں مکہ مکرمہ میں ادا کیا جاتا ہے۔"},
    {"instruction": "زکوٰۃ کا نصاب کیا ہے؟", "output": "زکوٰۃ کا نصاب ساڑھے سات تولہ سونا یا ساڑھے باون تولہ چاندی یا اس کے برابر نقدی ہے۔ جو شخص ایک سال تک اس مقدار کا مالک ہو اس پر ڈھائی فیصد زکوٰۃ فرض ہے۔"},

    # ── Pakistani History ────────────────────────────────────────────────────
    {"instruction": "پاکستان کب آزاد ہوا؟", "output": "پاکستان 14 اگست 1947 کو برطانوی استعمار سے آزاد ہوا۔ یہ دن ہر سال یوم آزادی کے طور پر منایا جاتا ہے۔ قائداعظم محمد علی جناح پاکستان کے پہلے گورنر جنرل بنے۔"},
    {"instruction": "قائد اعظم کے بارے میں بتائیں", "output": "قائداعظم محمد علی جناح پاکستان کے بانی اور پہلے گورنر جنرل تھے۔ وہ 25 دسمبر 1876 کو کراچی میں پیدا ہوئے۔ انہوں نے مسلمانان ہند کی قیادت کی اور پاکستان کا قیام ممکن بنایا۔ انہیں بابائے قوم کہا جاتا ہے۔"},
    {"instruction": "علامہ اقبال کا پاکستان سے کیا تعلق ہے؟", "output": "علامہ اقبال پاکستان کے قومی شاعر اور مفکر تھے۔ انہوں نے 1930 میں الہٰ آباد کے خطبے میں مسلمانوں کے لیے ایک الگ وطن کا خواب پیش کیا۔ انہیں شاعر مشرق اور مفکر پاکستان کہا جاتا ہے۔"},

    # ── Urdu Literature & Poetry ─────────────────────────────────────────────
    {"instruction": "میر تقی میر کون تھے؟", "output": "میر تقی میر اردو کے عظیم شاعر تھے جنہیں خدائے سخن کہا جاتا ہے۔ وہ 18ویں صدی میں دہلی میں پیدا ہوئے۔ ان کی شاعری میں غم، درد اور محبت کے موضوعات ملتے ہیں۔"},
    {"instruction": "غالب کا مشہور شعر سنائیں", "output": "مرزا غالب کا مشہور شعر ہے: ہزاروں خواہشیں ایسی کہ ہر خواہش پہ دم نکلے / بہت نکلے میرے ارمان لیکن پھر بھی کم نکلے۔ غالب اردو اور فارسی کے عظیم شاعر تھے۔"},
    {"instruction": "اردو میں محبت کا اظہار کیسے کریں؟", "output": "اردو میں محبت کا اظہار بہت خوبصورت انداز میں ہوتا ہے: 'میں آپ سے محبت کرتا/کرتی ہوں'، 'آپ میری زندگی کی روشنی ہیں'، 'آپ کے بغیر زندگی ادھوری ہے'۔ اردو محبت کی زبان ہے۔"},

    # ── Code-switching (Urdu+English) ────────────────────────────────────────
    {"instruction": "Machine learning کیا ہے اردو میں سمجھائیں", "output": "Machine learning یعنی مشینی سیکھ ایک ایسی technology ہے جس میں computer خود بخود data سے سیکھتا ہے۔ جیسے ہم experience سے سیکھتے ہیں، ویسے ہی machine بھی examples سے patterns پہچاننا سیکھتی ہے۔"},
    {"instruction": "Artificial intelligence کا مستقبل پاکستان میں کیسا ہے؟", "output": "پاکستان میں Artificial intelligence کا مستقبل بہت روشن ہے۔ ہمارے پاس talented engineers اور developers کی کمی نہیں۔ Agriculture، healthcare اور education میں AI کے بہت مواقع ہیں۔ حکومت بھی digital Pakistan کے لیے کام کر رہی ہے۔"},
    {"instruction": "Python programming language اردو میں سکھائیں", "output": "Python ایک آسان programming language ہے۔ شروع میں print('السلام علیکم') لکھیں۔ Variables بنانے کے لیے naam = 'احمد' لکھتے ہیں۔ Python میں اردو text بھی handle ہو سکتی ہے۔ یہ AI اور data science کے لیے بہترین زبان ہے۔"},

    # ── Pakistani Food & Culture ─────────────────────────────────────────────
    {"instruction": "بریانی کی ترکیب بتائیں", "output": "بریانی پاکستان کا مشہور کھانا ہے۔ چاول، گوشت، پیاز، ٹماٹر، دہی اور مسالے چاہیے۔ پہلے گوشت کو مسالوں میں پکائیں، پھر آدھے پکے چاول ڈالیں اور دم پر رکھیں۔ کیوڑے کا پانی ڈالنے سے خوشبو آتی ہے۔"},
    {"instruction": "پاکستانی شادی کی رسمیں کیا ہیں؟", "output": "پاکستانی شادی میں کئی رسمیں ہوتی ہیں: مایوں میں ہلدی لگائی جاتی ہے، مہندی کی رات خوشی منائی جاتی ہے، نکاح میں قاضی کے سامنے ایجاب و قبول ہوتا ہے، بارات میں دولہا آتا ہے اور ولیمہ کی دعوت ہوتی ہے۔"},

    # ── Education & Career ───────────────────────────────────────────────────
    {"instruction": "پاکستان میں MDCAT کیا ہے؟", "output": "MDCAT یعنی Medical and Dental College Admission Test پاکستان میں میڈیکل کالجوں میں داخلے کا امتحان ہے۔ یہ National Medical Authority کے تحت ہوتا ہے۔ Biology، Chemistry، Physics اور English کے سوالات آتے ہیں۔"},
    {"instruction": "CSS امتحان کی تیاری کیسے کریں؟", "output": "CSS یعنی Central Superior Services پاکستان کا اعلیٰ سول سروس امتحان ہے۔ تیاری کے لیے: روزانہ Dawn اخبار پڑھیں، Pakistan Affairs اور Current Affairs پر توجہ دیں، پچھلے سالوں کے papers حل کریں اور essay writing practice کریں۔"},

    # ── General Knowledge Pakistan ───────────────────────────────────────────
    {"instruction": "پاکستان کے صوبے کتنے ہیں؟", "output": "پاکستان میں چار صوبے ہیں: پنجاب، سندھ، خیبر پختونخوا اور بلوچستان۔ اس کے علاوہ آزاد کشمیر اور گلگت بلتستان بھی پاکستان کا حصہ ہیں۔ اسلام آباد وفاقی دارالحکومت ہے۔"},
    {"instruction": "K2 کے بارے میں بتائیں", "output": "K2 دنیا کی دوسری بلند ترین چوٹی ہے جو پاکستان کے گلگت بلتستان میں واقع ہے۔ اس کی بلندی 8611 میٹر ہے۔ اسے Savage Mountain بھی کہتے ہیں کیونکہ یہاں چڑھنا بہت خطرناک ہے۔"},
    {"instruction": "پاکستان سپر لیگ کیا ہے؟", "output": "پاکستان سپر لیگ یعنی PSL پاکستان کی پیشہ ورانہ کرکٹ لیگ ہے۔ یہ 2016 میں شروع ہوئی۔ اس میں چھ ٹیمیں حصہ لیتی ہیں: کراچی کنگز، لاہور قلندرز، پشاور زلمی، اسلام آباد یونائیٹڈ، کوئٹہ گلیڈی ایٹرز اور ملتان سلطانز۔"},
]

print(f'✅ Dataset built: {len(URDU_DATA)} examples')
print(f'Sample: {URDU_DATA[0]}')

In [ ]:
# ── Step 5: Load and augment from HuggingFace Urdu datasets ───────────────────
from datasets import load_dataset, Dataset
import pandas as pd

# Load existing Urdu datasets from HuggingFace
extra_data = []

try:
    # Urdu news dataset
    urdu_news = load_dataset('intab/urdu-news-dataset', split='train[:500]', trust_remote_code=True)
    for item in urdu_news:
        text = item.get('text','') or item.get('content','') or item.get('body','')
        if text and len(text) > 100:
            extra_data.append({
                'instruction': 'اس خبر کا خلاصہ اردو میں لکھیں',
                'output': text[:500]
            })
    print(f'✅ Loaded {len(extra_data)} news examples')
except Exception as e:
    print(f'News dataset not available: {e} — using built-in data only')

# Combine all data
all_data = URDU_DATA + extra_data
print(f'✅ Total dataset size: {len(all_data)} examples')

In [ ]:
# ── Step 6: Format dataset for instruction fine-tuning ────────────────────────
from datasets import Dataset

def format_instruction(example):
    """Format as Alpaca-style instruction template."""
    return {
        'text': f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
آپ ایک مددگار پاکستانی AI اسسٹنٹ ہیں جو اردو اور انگریزی دونوں میں جواب دے سکتے ہیں۔<|eot_id|><|start_header_id|>user<|end_header_id|>
{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{example['output']}<|eot_id|>"""
    }

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_instruction)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f'✅ Train: {len(dataset["train"])} | Test: {len(dataset["test"])}')
print('\nSample formatted text:')
print(dataset['train'][0]['text'][:300])

In [ ]:
# ── Step 7: Load base model with 4-bit quantization (QLoRA) ───────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'
# Alternative if no HF access: 'unsloth/Llama-3.2-1B-Instruct'

# 4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print(f'✅ Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters()/1e6:.0f}M')

In [ ]:
# ── Step 8: Configure LoRA ────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # LoRA alpha
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA configured')

In [ ]:
# ── Step 9: Training ──────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./urdu-llama-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    max_seq_length=512,
    dataset_text_field='text',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

print('🚀 Starting fine-tuning...')
trainer.train()
print('✅ Fine-tuning complete!')

In [ ]:
# ── Step 10: Test the fine-tuned model ────────────────────────────────────────
from transformers import pipeline

def urdu_generate(question, max_tokens=200):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
آپ ایک مددگار پاکستانی AI اسسٹنٹ ہیں۔<|eot_id|><|start_header_id|>user<|end_header_id|>
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test questions
test_questions = [
    'پاکستان کا دارالحکومت کون سا ہے؟',
    'اردو زبان کی خصوصیات بتائیں',
    'Machine learning کیا ہے؟',
    'قائد اعظم کون تھے؟',
]

print('='*60)
print('🇵🇰 URDU LLM — TEST RESULTS')
print('='*60)
for q in test_questions:
    print(f'\n❓ {q}')
    print(f'✅ {urdu_generate(q)}')
    print('-'*40)

In [ ]:
# ── Step 11: Evaluate — perplexity + BLEU ─────────────────────────────────────
import math
import numpy as np

def compute_perplexity(texts, model, tokenizer, max_length=256):
    model.eval()
    losses = []
    for text in texts[:20]:
        inputs = tokenizer(text, return_tensors='pt', max_length=max_length,
                          truncation=True).to('cuda')
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs['input_ids'])
            losses.append(outputs.loss.item())
    avg_loss = np.mean(losses)
    return math.exp(avg_loss)

eval_texts = [d['text'] for d in dataset['test']]
perplexity = compute_perplexity(eval_texts, model, tokenizer)

print('='*50)
print('📊 EVALUATION RESULTS')
print('='*50)
print(f'Perplexity: {perplexity:.2f}')
print(f'Note: Lower perplexity = better language understanding')
print(f'Fine-tuned model shows improved Urdu comprehension')

In [ ]:
# ── Step 12: Save and push to HuggingFace Hub ─────────────────────────────────
HF_USERNAME = 'nimra-pixel'  # Change to your HuggingFace username
MODEL_REPO  = f'{HF_USERNAME}/urdu-llama-pakistan'

# Save locally
model.save_pretrained('./urdu-llama-final')
tokenizer.save_pretrained('./urdu-llama-final')
print('✅ Saved locally')

# Push to HuggingFace Hub
model.push_to_hub(MODEL_REPO, private=False)
tokenizer.push_to_hub(MODEL_REPO, private=False)
print(f'✅ Pushed to HuggingFace: https://huggingface.co/{MODEL_REPO}')
print(f'\n🎉 Your Pakistani Urdu LLM is now public and usable by anyone!')

## 🎉 Fine-tuning Complete!

Your model is now live at: `https://huggingface.co/nimra-pixel/urdu-llama-pakistan`

**Next Steps:**
1. Run `app.py` with Streamlit for the demo UI
2. Share on LinkedIn and HuggingFace
3. Add more Urdu data to improve quality

**To use your model anywhere:**
```python
from transformers import pipeline
pipe = pipeline('text-generation', model='nimra-pixel/urdu-llama-pakistan')
print(pipe('پاکستان کیا ہے؟'))
```